In [ ]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.preprocessing import StandardScaler 
import xgboost as xgb 
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
x = pd.read_csv(r'C:\Users\ALOK\Documents\TechIndians\X_features.csv')
y_reaction = pd.read_csv(r'C:\Users\ALOK\Documents\TechIndians\y_reaction.csv')
y_severity = pd.read_csv(r'C:\Users\ALOK\Documents\TechIndians\y_severity.csv')

In [ ]:
y_reaction.shape,y_severity.shape,x.shape

In [ ]:
df_model = x.copy()
df_model['y_reaction'] = y_reaction.values
df_model['y_severity']= y_severity.values 

In [ ]:
X_fea= df_model.drop(['y_reaction','y_severity'],axis = 1)
y_reaction_target = df_model['y_reaction'] 
y_severity_target = df_model['y_severity']-1

In [ ]:
# Y_Reaction (Reaction Type Prediction)
# What it predicts: Which specific adverse reaction will occur (e.g., "Nausea", "Headache", "Rash")

# Intuition/Use Case:

# Early warning system: "If patient takes Drug X for Condition Y, they're likely to experience Reaction Z"
# Personalized medicine: Predict which specific side effects a patient might experience
# Drug development: Identify common reaction patterns for new drugs
# Clinical decision support: Help doctors anticipate and prepare for specific adverse events
# Example: Patient on METFORMIN for diabetes ? Predict they'll likely experience "Gastrointestinal discomfort" rather than "Liver damage"

# Y_Severity (Severity Level Prediction)
# What it predicts: How serious the adverse event will be (1=Other, 3=Hospitalization, 6=Death)

# Intuition/Use Case:

# Risk stratification: Identify high-risk patients who need closer monitoring
# Resource allocation: Predict which cases need immediate intervention
# Drug safety: Flag drug combinations that lead to severe outcomes
# Regulatory decisions: Inform drug approval/withdrawal decisions
# Example: Same patient on METFORMIN ? Predict severity level 1 (mild, outpatient management) vs level 6 (life-threatening)

# Why Both?
# They answer different clinical questions:
# Reaction (WHAT): "What will happen to the patient?"
# Severity (HOW BAD): "How serious will it be?"
# Real-world scenario:

# Drug A ? 90% chance of nausea (annoying but manageable)
# Drug B ? 10% chance of liver failure (rare but deadly)

In [ ]:
# Split data (70% train, 15% validation, 15% test)
X_train, X_temp, y_reaction_train, y_reaction_temp = train_test_split(
    X_fea, y_reaction_target, test_size=0.3, random_state=42, stratify=y_reaction_target
)

X_val, X_test, y_reaction_val, y_reaction_test = train_test_split(
    X_temp, y_reaction_temp, test_size=0.5, random_state=42, stratify=y_reaction_temp
)

# Same split for severity
_, _, y_severity_train, y_severity_temp = train_test_split(
    X_fea, y_severity_target, test_size=0.3, random_state=42, stratify=y_severity_target
)

_, _, y_severity_val, y_severity_test = train_test_split(
    X_temp, y_severity_temp, test_size=0.5, random_state=42, stratify=y_severity_temp
)

In [ ]:
print(f"\nTrain set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### reaction classifier(not needed)

In [ ]:
xgb_reaction = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_reaction.fit(
    X_train_scaled, y_reaction_train,
    eval_set=[(X_val_scaled, y_reaction_val)],
    verbose=True
)

### severity classifier (important)

In [ ]:
xgb_severity = xgb.XGBClassifier(
    n_estimators=130,
    max_depth=10,
    learning_rate=0.1436348834661225,
    subsample=0.6109356180111788,
    colsample_bytree=0.9448531875640296,
    min_child_weight=7,
    gamma=4.260249158096812,

    # required / good practice
    objective="multi:softprob",
    num_class=6,
    eval_metric="mlogloss",
    random_state=42
)

# Train the model
xgb_severity.fit(
    X_train_scaled, y_severity_train,
    eval_set=[(X_val_scaled, y_severity_val)],
    verbose=True
)

In [ ]:
y_severity_pred_xgb = xgb_severity.predict(X_test)

# Evaluate
print(f"\nXGBoost Severity Prediction Accuracy: {accuracy_score(y_severity_test, y_severity_pred_xgb):.4f}")
print("\nClassification Report:")
print(classification_report(y_severity_test, y_severity_pred_xgb))

In [ ]:
import matplotlib.pyplot as plt
import xgboost as xgb

plt.figure(figsize=(8, 6))

xgb.plot_importance(
    xgb_severity,
    max_num_features=13
)

plt.title('XGBoost Feature Importance - Severity Prediction')
plt.tight_layout()
plt.savefig('xgboost_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
xgb_severity.save_model("xgb_severity_model.json")

In [ ]:
import json

params = xgb_severity.get_params()

with open("xgb_severity_params.json", "w") as f:
    json.dump(params, f, indent=4)